## Import Data

In [ ]:
import os
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import gaussian_filter
def get_data(all_sessions_path, dt, gc_or_not=True):
    # gc = True si A1, si PMC alors False
    # Initialisation
    n_data_s = []
    f_data_s = []
    neuron_ids_s = []
    global_id = 0

    for file in tqdm(all_sessions_path):
        print(file)
        try:
            # Chargement
            n_data = np.load(os.path.join(file, f'data_{dt}.npy'))
            f_data_raw = np.load(os.path.join(file, f'features_{dt}.npy'), allow_pickle=True)
            if gc_or_not:
                gc = np.load(os.path.join(file, 'good_clusters.npy'))
            else:
                gc = np.arange(len(n_data))
            unique_tones_path = r"/home/skieur/Documents/Data/Bohan/unique_tones/unique_tones.npy"
            try:
                unique_tones = np.load(unique_tones_path)
            except FileNotFoundError:
                print(f"Unique tones file not found at path: {unique_tones_path}")
                raise
            # Neurones valides
            n_data = n_data[gc, :].astype(float)

            # Construction DataFrame
            f_data_dict = {
                'Played_frequency': [],
                'Condition': [],
                'Block': [],
                'Frequency_changes': [],
                'Mock_frequency': [],
                'Mock_change': []
            }
            for item in f_data_raw:
                for key, value in item.items():
                    f_data_dict[key].append(value)
            f_data = pd.DataFrame(f_data_dict)

            # IDs des neurones
            N_neurons = len(n_data)
            neuron_ids = list(range(global_id, global_id + N_neurons))
            global_id += N_neurons
            neuron_ids_s.append(neuron_ids)

            # Direction
            #f_data['Change_direction'] = compare_diff(f_data['Played_frequency'].to_numpy())
            #f_data['Mock_direction'] = compare_diff(f_data['Mock_frequency'].to_numpy())

            # --- mapping des fréquences vers pixels ---
            unique_tones_sorted = np.sort(unique_tones)
            pixels_sorted = np.linspace(0, 28, len(unique_tones_sorted))  # 28 cm

            # --- choisir Played ou Mock selon la condition ---
            positional_freq = np.where(
                f_data['Condition'].isin([0, -1]),
                f_data['Played_frequency'],
                f_data['Mock_frequency']
            )

            # --- interpolation pour toutes les fréquences inconnues ---
            positions = np.interp(positional_freq, unique_tones_sorted, pixels_sorted)

            # --- lissage ---
            pos_smooth = gaussian_filter(positions, sigma=10)

            # --- calcul de Speed_x ---
            speed_x = np.diff(pos_smooth)
            speed_x = np.append(0, speed_x)
            speed_x = np.abs(speed_x) * 100
            speed_x[~np.isfinite(speed_x)] = 0  # supprime NaN / inf
            f_data['Speed_x'] = speed_x

            # --- calcul de Sound_speed ---
            freq = np.array(f_data['Played_frequency'])
            freq_pos = np.interp(freq, unique_tones_sorted, pixels_sorted)
            freq_pos_smooth = gaussian_filter(freq_pos, sigma=10)
            sound_speed = np.diff(freq_pos_smooth)
            sound_speed = np.append(0, sound_speed)
            sound_speed = np.abs(sound_speed) * 100
            sound_speed[~np.isfinite(sound_speed)] = 0
            f_data['Sound_speed'] = sound_speed
            f_data['Position'] = pos_smooth
            f_data['Freq_position'] = freq_pos_smooth  # les positions correspondantes aux fréquences jouées

            speed_bins = [0, 0.01, 1, 2, 3, 4, 5, 6, np.inf]
            f_data['Speed_bin'] = pd.cut(
                f_data['Speed_x'],
                bins=speed_bins,
                labels=False,
                include_lowest=True
            )
            acc_x = np.diff(speed_x, prepend=speed_x[0])
            f_data["Acc_x"] = acc_x

            # smooth n_data
            n_data_o = n_data - n_data.mean(axis=1, keepdims=True)
            n_data_smooth = gaussian_filter(n_data_o, sigma=1, axes=1)
            # Sauvegarde
            n_data_s.append(n_data_smooth)
            f_data_s.append(f_data)

        except Exception as e:
            print(f"Error for file {file}: {e}")
    return n_data_s, f_data_s

In [ ]:
import os

base_path = r"/home/skieur/Documents/Data/Bohan"

all_sessions_path = [
    os.path.join(base_path, name) 
    for name in os.listdir(base_path) 
    if os.path.isdir(os.path.join(base_path, name))
]

all_sessions_path[:2]

## Set Up Essential Variables

In [ ]:

dt = 0.005
#gc_or_not = True (if you only want the good clusters... keep it True)
n_data_s, f_data_s = get_data(all_sessions_path, dt, gc_or_not= True)


In [ ]:
# 1. Setup Parameters
t_window_pre = 0.32
t_window_post = 0.22

n_bins_pre = int(t_window_pre / dt)
n_bins_post = int(t_window_post / dt)
expected_length = n_bins_pre + n_bins_post


## Overall Average

In [ ]:

# List to collect individual NEURON means (PSTHs) from all sessions
all_neuron_means = []

# 2. Loop through all sessions 
for n_data, f_data in zip(n_data_s, f_data_s):
    
    event_playback = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 1))
    
    # Store events specifically for THIS session
    session_events = [] 
    
    for i in event_playback:
        start = int(i - n_bins_pre)
        stop = int(i + n_bins_post)
        
        if start >= 0 and stop <= n_data.shape[-1]:
            # segment shape: (n_channels, expected_length)
            segment = n_data[:, start:stop]
            if segment.shape[1] == expected_length:
                session_events.append(segment)
    
    # If we found valid events in this session
    if len(session_events) > 0:
        # Stack to shape: (n_events, n_channels, expected_length)
        session_stack = np.stack(session_events, axis=0)
        
        # STEP 1: Average across multiple events (axis=0)
        # Resulting shape: (n_channels, expected_length)
        # This represents the average trace for each individual channel in this session
        session_channel_means = np.mean(session_stack, axis=0)
        
        # Append these channel means to our global list
        all_neuron_means.append(session_channel_means)

# 3. Combine all 16 sessions * ~32 channels and Calculate Grand Average
if len(all_neuron_means) > 0:
    # Concatenate along axis=0 (the channel dimension)
    # Using concatenate because one session might have 32 channels, another 31, etc.
    # Resulting shape: (Total_Channels_Across_All_Sessions, expected_length)
    final_pool = np.concatenate(all_neuron_means, axis=0)
    
    # STEP 2: Calculate the population mean across all 16*32 channels
    # Resulting shape: (expected_length,)
    mean_playback = np.mean(final_pool, axis=0)
    
    # Calculate Standard Error (SEM) based on the number of CHANNELS, not events
    sem_playback = np.std(final_pool, axis=0) / np.sqrt(final_pool.shape[0])
    
    print(f"Successfully collected averaged traces from {final_pool.shape[0]} individual channels/neurons.")
    print(f"Mean trajectory calculated with shape: {mean_playback.shape}")
else:
    print("No valid events found.")

In [ ]:
mean_playback

In [ ]:


# 1. Create the time axis
# Ensure the length matches mean_playback (n_bins_pre + n_bins_post)
time = np.arange(-n_bins_pre, n_bins_post) * dt

# 2. Compatibility check: 
# Option 1 output 'mean_playback' is already a 1D array (the Global Average).
# We can wrap it in a Series or DataFrame for easy handling.
df_plot = pd.DataFrame({'Global_Avg': mean_playback}, index=time)

# 3. Plotting
plt.figure(figsize=(8, 5))

# Plot the 1D mean directly
plt.plot(df_plot.index, df_plot['Global_Avg'], 
         color='black', linewidth=2.5, label='Block 5 Playback (Global Avg)')

# 4. Formatting
plt.axvline(0, color='black', linestyle='--', label='Event')
plt.title(f'Playback Average Neural Response (Pooled Across {final_pool.shape[0]} neuron samples)', fontsize=14)
plt.xlabel('Time relative to event (s)', fontsize=12)
plt.ylabel('Mean Activity (Firing Rate)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

# Save or display
# plt.savefig('global_average_response.png')
plt.show()

## Average of Tracking vs. Playback 

In [ ]:
# Lists to collect means for each neuron/channel from all sessions
all_neuron_means_playback = []
all_neuron_means_tracking = []

# 2. Loop through all sessions 
for n_data, f_data in zip(n_data_s, f_data_s):
    
    # Get event indices for both conditions
    event_playback = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 1))
    event_tracking = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 0))
    
    # --- Helper function: extract average response for current session ---
    def get_session_channel_means(event_indices):
        session_events = []
        for i in event_indices:
            start = int(i - n_bins_pre)
            stop = int(i + n_bins_post)
            if start >= 0 and stop <= n_data.shape[-1]:
                segment = n_data[:, start:stop]
                if segment.shape[1] == expected_length:
                    session_events.append(segment)
        if len(session_events) > 0:
            session_stack = np.stack(session_events, axis=0)
            return np.mean(session_stack, axis=0) # shape: (n_channels, expected_length)
        return None
    # -------------------------------------------------------------
    
    # Extract Playback and append to list
    pb_means = get_session_channel_means(event_playback)
    if pb_means is not None:
        all_neuron_means_playback.append(pb_means)
        
    # Extract Tracking and append to list
    tr_means = get_session_channel_means(event_tracking)
    if tr_means is not None:
        all_neuron_means_tracking.append(tr_means)

# 3. Combine and Calculate Grand Averages for BOTH conditions
# Playback
if len(all_neuron_means_playback) > 0:
    final_pool_pb = np.concatenate(all_neuron_means_playback, axis=0)
    mean_playback = np.mean(final_pool_pb, axis=0)
    sem_playback = np.std(final_pool_pb, axis=0) / np.sqrt(final_pool_pb.shape[0])
    print(f"Playback: {final_pool_pb.shape[0]} neurons pooled.")

# Tracking
if len(all_neuron_means_tracking) > 0:
    final_pool_tr = np.concatenate(all_neuron_means_tracking, axis=0)
    mean_tracking = np.mean(final_pool_tr, axis=0)
    sem_tracking = np.std(final_pool_tr, axis=0) / np.sqrt(final_pool_tr.shape[0])
    print(f"Tracking: {final_pool_tr.shape[0]} neurons pooled.")

In [ ]:
mean_playback

In [ ]:
# Create time axis
time = np.arange(-n_bins_pre, n_bins_post) * dt

plt.figure(figsize=(9, 6))

# Plot Playback curve and shaded error
plt.plot(time, mean_playback, color='black', linewidth=2.5, label='Playback')
plt.fill_between(time, 
                 mean_playback - sem_playback, 
                 mean_playback + sem_playback, 
                 color='black', alpha=0.2, label='SEM-Playback')

# Plot Tracking curve and shaded error
plt.plot(time, mean_tracking, color='red', linewidth=2.5, label='Tracking')
plt.fill_between(time, 
                 mean_tracking - sem_tracking, 
                 mean_tracking + sem_tracking, 
                 color='red', alpha=0.2, label='SEM-Tracking')

# Figure decorations
plt.axvline(0, color='black', linestyle='--', label='Event Trigger')
plt.title(f'Population Neural Response: Playback ({final_pool_pb.shape[0]} neurons) vs Tracking ({final_pool_tr.shape[0]} neurons) (bin = {dt}s)', fontsize=14)
plt.xlabel('Time relative to event (s)', fontsize=12)
plt.ylabel('Mean Activity (Firing Rate)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
# Iterate through the list of DataFrames (f_data_s)
for i, f_data in enumerate(f_data_s):
    # Get the session folder name from the path list if available
    session_name = os.path.basename(all_sessions_path[i])
    
    print(f"--- Session {i}: {session_name} ---")
    
    # Check if 'Block' column exists
    if 'Block' in f_data.columns:
        # Sort by block index (e.g., Block 1, 2, 3...)
        block_counts = f_data['Block'].value_counts().sort_index()
        print(block_counts)
    else:
        print("Column 'Block' not found in this session.")
    print("\n")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Containers for the 4 groups
# Format: (First/Last) Block x (Playback/Tracking) Condition
data_groups = {
    'first_playback': [], 'first_tracking': [],
    'last_playback': [], 'last_tracking': []
}

# 2. Helper function to get channel means for specific criteria
def get_channel_means(n_data, f_data, block_id, condition_id):
    events = np.flatnonzero((f_data['Block'] == block_id) & 
                           (f_data['Condition'] == condition_id) & 
                           (f_data['Frequency_changes'] == 1))
    
    segments = []
    for i in events:
        start, stop = int(i - n_bins_pre), int(i + n_bins_post)
        if 0 <= start and stop <= n_data.shape[-1]:
            seg = n_data[:, start:stop]
            if seg.shape[1] == expected_length:
                segments.append(seg)
    
    if len(segments) > 0:
        return np.mean(np.stack(segments), axis=0) # Mean per channel
    return None

# 3. Main Processing Loop
for n_data, f_data in zip(n_data_s, f_data_s):
    # Find blocks that have BOTH condition 0 and 1
    blocks_with_both = []
    for b in sorted(f_data['Block'].unique()):
        has_cond0 = not f_data[(f_data['Block'] == b) & (f_data['Condition'] == 0)].empty
        has_cond1 = not f_data[(f_data['Block'] == b) & (f_data['Condition'] == 1)].empty
        if has_cond0 and has_cond1:
            blocks_with_both.append(b)
            
    if len(blocks_with_both) < 1:
        continue # Skip session if no block meets criteria
        
    first_b = blocks_with_both[0]
    last_b = blocks_with_both[-1]
    
    # Extract for First Block
    fp = get_channel_means(n_data, f_data, first_b, 1)
    ft = get_channel_means(n_data, f_data, first_b, 0)
    if fp is not None: data_groups['first_playback'].append(fp)
    if ft is not None: data_groups['first_tracking'].append(ft)
        
    # Extract for Last Block (only if it's different, or include anyway)
    lp = get_channel_means(n_data, f_data, last_b, 1)
    lt = get_channel_means(n_data, f_data, last_b, 0)
    if lp is not None: data_groups['last_playback'].append(lp)
    if lt is not None: data_groups['last_tracking'].append(lt)

# 4. Calculate Final Means and SEMs
results = {}
for key, list_of_means in data_groups.items():
    if list_of_means:
        combined = np.concatenate(list_of_means, axis=0)
        results[key] = {
            'mean': np.mean(combined, axis=0),
            'sem': np.std(combined, axis=0) / np.sqrt(combined.shape[0])
        }



In [ ]:
last_b

In [ ]:
results = {}
for key, list_of_means in data_groups.items():
    if list_of_means:
        combined = np.concatenate(list_of_means, axis=0)
        results[key] = {
            'mean': np.mean(combined, axis=0),
            'sem': np.std(combined, axis=0) / np.sqrt(combined.shape[0]),
            'count': combined.shape[0]  # <--- 关键：把神经元数量存下来
        }

# 创建两个并排的子图
fig, axes = plt.subplots(1, 2, figsize=(16, 10), sharey=0)

# --- 左图：First Block ---
ax1 = axes[0]
n_pb_first = results['first_playback']['count']
n_tr_first = results['first_tracking']['count']

ax1.plot(time, results['first_playback']['mean'], color='black', linewidth=2.5, 
         label=f'Playback (n={n_pb_first})') # 在图例显示数量
ax1.plot(time, results['first_tracking']['mean'], color='red', linewidth=2.5, 
         label=f'Tracking (n={n_tr_first})')
ax1.fill_between(time, 
                 results['first_tracking']['mean'] - results['first_tracking']['sem'], 
                 results['first_tracking']['mean'] + results['first_tracking']['sem'], 
                 color='red', alpha=0.2, label='SEM-Tracking')
ax1.fill_between(time, 
                 results['first_playback']['mean'] - results['first_playback']['sem'], 
                 results['first_playback']['mean'] + results['first_playback']['sem'], 
                 color='black', alpha=0.2, label='SEM-Playback')                 
ax1.set_title(f'First Block\n(n={n_tr_first} neurons)', fontsize=14)


# --- 右图：Last Block ---
ax2 = axes[1]
n_pb_last = results['last_playback']['count']
n_tr_last = results['last_tracking']['count']

ax2.plot(time, results['last_playback']['mean'], color='black', linewidth=2.5, 
         label=f'Playback (n={n_pb_last})')
ax2.plot(time, results['last_tracking']['mean'], color='red', linewidth=2.5, 
         label=f'Tracking (n={n_tr_last})')
ax2.fill_between(time, 
                 results['last_tracking']['mean'] - results['last_tracking']['sem'], 
                 results['last_tracking']['mean'] + results['last_tracking']['sem'], 
                 color='red', alpha=0.2)
ax2.fill_between(time, 
                 results['last_playback']['mean'] - results['last_playback']['sem'], 
                 results['last_playback']['mean'] + results['last_playback']['sem'], 
                 color='black', alpha=0.2)

ax2.set_title(f'Last Block\n(n={n_tr_last} neurons)', fontsize=14)


# --- 总标题修改 ---
# 如果你想在总标题显示总数，可以直接取一个代表性的数字
plt.suptitle(f'Neural Response Comparison: Tracking (Red) vs Playback (Black) (bin = {dt}s)', 
             fontsize=16, y=1.05)

# --- 通用格式设置 ---
for ax in axes:
    ax.axvline(0, color='gray', linestyle='--', alpha=0.7, label='Event')
    ax.set_xlabel('Time relative to event (s)', fontsize=12)
    ax.set_ylabel('Firing Rate', fontsize=12)
    ax.grid(True, alpha=0.2)
    ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:

# 1. 计算差值 (Last - First) 和 误差传递 (Error Propagation)
# Playback Difference
diff_pb_mean = results['last_playback']['mean'] - results['first_playback']['mean']
diff_pb_sem = np.sqrt(results['first_playback']['sem']**2 + results['last_playback']['sem']**2)

# Tracking Difference
diff_tr_mean = results['last_tracking']['mean'] - results['first_tracking']['mean']
diff_tr_sem = np.sqrt(results['first_tracking']['sem']**2 + results['last_tracking']['sem']**2)

# 2. 设置画布：3行1列
fig, axes = plt.subplots(3, 1, figsize=(10, 18), sharex=True)

# --- Top Plot: First Block ---
ax1 = axes[0]
ax1.plot(time, results['first_playback']['mean'], color='black', linewidth=2, label='Playback')
ax1.fill_between(time, results['first_playback']['mean'] - results['first_playback']['sem'], 
                 results['first_playback']['mean'] + results['first_playback']['sem'], color='black', alpha=0.1)
ax1.plot(time, results['first_tracking']['mean'], color='red', linewidth=2, label='Tracking')
ax1.fill_between(time, results['first_tracking']['mean'] - results['first_tracking']['sem'], 
                 results['first_tracking']['mean'] + results['first_tracking']['sem'], color='red', alpha=0.1)
ax1.set_title(f"First Common Block (n={results['first_tracking']['count']} neurons)", fontsize=14)

# --- Middle Plot: Last Block ---
ax2 = axes[1]
ax2.plot(time, results['last_playback']['mean'], color='black', linewidth=2, label='Playback')
ax2.fill_between(time, results['last_playback']['mean'] - results['last_playback']['sem'], 
                 results['last_playback']['mean'] + results['last_playback']['sem'], color='black', alpha=0.1)
ax2.plot(time, results['last_tracking']['mean'], color='red', linewidth=2, label='Tracking')
ax2.fill_between(time, results['last_tracking']['mean'] - results['last_tracking']['sem'], 
                 results['last_tracking']['mean'] + results['last_tracking']['sem'], color='red', alpha=0.1)
ax2.set_title(f"Last Common Block (n={results['last_tracking']['count']} neurons)", fontsize=14)

# --- Bottom Plot: Difference (Last - First) ---
ax3 = axes[2]
# 绘制 Playback 差值
ax3.plot(time, diff_pb_mean, color='black', linewidth=2.5, label='Playback Difference')
# ax3.fill_between(time, diff_pb_mean - diff_pb_sem, diff_pb_mean + diff_pb_sem, color='black', alpha=0.1)

# 绘制 Tracking 差值
ax3.plot(time, diff_tr_mean, color='red', linewidth=2.5, label='Tracking Difference')
# ax3.fill_between(time, diff_tr_mean - diff_tr_sem, diff_tr_mean + diff_tr_sem, color='red', alpha=0.1)

# 在差值图增加 y=0 的水平基准线，方便看增减
ax3.axhline(0, color='blue', linestyle=':', alpha=0.5)
ax3.set_title("Difference (Last Block - First Block)", fontsize=14)
ax3.set_ylabel('$\Delta$ Firing Rate', fontsize=12)

# --- 通用格式设置 ---
for ax in axes:
    ax.axvline(0, color='gray', linestyle='--', alpha=0.7)
    ax.set_xlabel('Time relative to event (s)', fontsize=12)
    ax.set_ylabel('Firing Rate', fontsize=12)
    ax.grid(True, alpha=0.2)
    ax.legend(loc='upper right', fontsize=10)

plt.suptitle(f'Neural Dynamics: First vs Last Block with Difference (bin = {dt}s)', fontsize=16, y=0.99)
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

In [ ]:
# 1. Setup Parameters (re-using your existing variables if already defined)
time = np.arange(-n_bins_pre, n_bins_post) * dt
n_triggers = 1  # Number of events to take from each condition for the subset analysis

# Lists to collect means for each channel across sessions
means_first_n_pb = []
means_last_n_tr = []

# 2. Helper function to extract and average segments for specific event indices
def get_subset_channel_means(n_data, event_indices):
    segments = []
    for i in event_indices:
        start, stop = int(i - n_bins_pre), int(i + n_bins_post)
        if 0 <= start and stop <= n_data.shape[-1]:
            seg = n_data[:, start:stop]
            if seg.shape[1] == expected_length:
                segments.append(seg)
                
    if len(segments) > 0:
        return np.mean(np.stack(segments, axis=0), axis=0) # Mean per channel
    return None

# 3. Process each session
for n_data, f_data in zip(n_data_s, f_data_s):
    # Get all valid events for Playback (Condition 1) and Tracking (Condition 0)
    events_pb = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 1))
    events_tr = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 0))
    
    # Subset: First 100 Playback, Last 100 Tracking
    events_pb_subset = events_pb[:n_triggers] if len(events_pb) >= n_triggers else events_pb
    events_tr_subset = events_tr[-n_triggers:] if len(events_tr) >= n_triggers else events_tr
    
    # Extract and store
    pb_mean = get_subset_channel_means(n_data, events_pb_subset)
    tr_mean = get_subset_channel_means(n_data, events_tr_subset)
    
    if pb_mean is not None: means_first_n_pb.append(pb_mean)
    if tr_mean is not None: means_last_n_tr.append(tr_mean)

# 4. Calculate Grand Averages
final_pb = np.concatenate(means_first_n_pb, axis=0)
final_tr = np.concatenate(means_last_n_tr, axis=0)

res_pb = {
    'mean': np.mean(final_pb, axis=0),
    'sem': np.std(final_pb, axis=0) / np.sqrt(final_pb.shape[0]),
    'count': final_pb.shape[0]
}

res_tr = {
    'mean': np.mean(final_tr, axis=0),
    'sem': np.std(final_tr, axis=0) / np.sqrt(final_tr.shape[0]),
    'count': final_tr.shape[0]
}

# Calculate Difference (Tracking - Playback)
diff_mean = res_tr['mean'] - res_pb['mean']
# Standard error of the difference
diff_sem = np.sqrt(res_tr['sem']**2 + res_pb['sem']**2)

# 5. Plotting
fig, axes = plt.subplots(2, 1, figsize=(10, 12), sharex=True)

# --- Top Plot: Overlaid Means ---
ax1 = axes[0]
ax1.plot(time, res_pb['mean'], color='black', linewidth=2.5, 
         label=f"First {n_triggers} Playback (n={res_pb['count']})")
ax1.fill_between(time, res_pb['mean'] - res_pb['sem'], res_pb['mean'] + res_pb['sem'], 
                 color='black', alpha=0.15)

ax1.plot(time, res_tr['mean'], color='red', linewidth=2.5, 
         label=f"Last {n_triggers} Tracking (n={res_tr['count']})")
ax1.fill_between(time, res_tr['mean'] - res_tr['sem'], res_tr['mean'] + res_tr['sem'], 
                 color='red', alpha=0.15)

ax1.set_title(f"Neural Response: Last {n_triggers} Tracking vs First {n_triggers} Playback", fontsize=14)
ax1.set_ylabel('Firing Rate', fontsize=12)

# --- Bottom Plot: Difference ---
ax2 = axes[1]
ax2.plot(time, diff_mean, color='purple', linewidth=2.5, label='Difference (Tracking - Playback)')
ax2.fill_between(time, diff_mean - diff_sem, diff_mean + diff_sem, color='purple', alpha=0.15)
ax2.axhline(0, color='gray', linestyle=':', linewidth=2, alpha=0.8)

ax2.set_title("Response Difference", fontsize=14)
ax2.set_xlabel('Time relative to event (s)', fontsize=12)
ax2.set_ylabel('$\Delta$ Firing Rate', fontsize=12)

# Format both axes
for ax in axes:
    ax.axvline(0, color='gray', linestyle='--', alpha=0.7, label='Event Trigger' if ax == ax1 else "")
    ax.grid(True, alpha=0.2)
    ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.ticker as ticker
# Define indices for calculating metrics
# Baseline: 0.2s before frequency change
baseline_bins = int(0.2 / dt)
idx_base_start = n_bins_pre - baseline_bins
idx_base_end = n_bins_pre  # Frequency change (time 0)

# Peak: Max response after frequency change
idx_peak_start = n_bins_pre
idx_peak_end = expected_length

# Define the different "n_triggers" (number of events) you want to evaluate
n_triggers_list = [1, 2, 3, 5, 7, 10, 20, 30, 50, 70, 100, 200, 300, 500]

# Dictionary to store the results
results_metrics = {
    'n_events': [],
    'pb_baseline': [], 'pb_peak': [], 'pb_evoked': [],
    'tr_baseline': [], 'tr_peak': [], 'tr_evoked': []
}

# Helper function to extract and average segments
def get_subset_channel_means(n_data, event_indices):
    segments = []
    for i in event_indices:
        start, stop = int(i - n_bins_pre), int(i + n_bins_post)
        if 0 <= start and stop <= n_data.shape[-1]:
            seg = n_data[:, start:stop]
            if seg.shape[1] == expected_length:
                segments.append(seg)
    if len(segments) > 0:
        return np.mean(np.stack(segments, axis=0), axis=0) 
    return None

# 2. Main Processing Loop
for n_b in n_triggers_list:
    means_first_n_pb = []
    means_last_n_tr = []
    
    for n_data, f_data in zip(n_data_s, f_data_s):
        events_pb = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 1))
        events_tr = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 0))
        
        # Subset: First n_b Playback, Last n_b Tracking
        events_pb_subset = events_pb[:n_b] if len(events_pb) >= n_b else events_pb
        events_tr_subset = events_tr[-n_b:] if len(events_tr) >= n_b else events_tr
        
        pb_mean = get_subset_channel_means(n_data, events_pb_subset)
        tr_mean = get_subset_channel_means(n_data, events_tr_subset)
        
        if pb_mean is not None: means_first_n_pb.append(pb_mean)
        if tr_mean is not None: means_last_n_tr.append(tr_mean)

    # Calculate Grand Averages for this n_b
    final_pb = np.concatenate(means_first_n_pb, axis=0)
    final_tr = np.concatenate(means_last_n_tr, axis=0)
    
    mean_pb_trace = np.mean(final_pb, axis=0)
    mean_tr_trace = np.mean(final_tr, axis=0)
    
    # --- Calculate Metrics for Playback ---
    pb_base = np.mean(mean_pb_trace[idx_base_start:idx_base_end])
    pb_pk = np.max(mean_pb_trace[idx_peak_start:idx_peak_end])
    pb_evk = pb_pk - pb_base
    
    # --- Calculate Metrics for Tracking ---
    tr_base = np.mean(mean_tr_trace[idx_base_start:idx_base_end])
    tr_pk = np.max(mean_tr_trace[idx_peak_start:idx_peak_end])
    tr_evk = tr_pk - tr_base
    
    # Store results
    results_metrics['n_events'].append(n_b)
    results_metrics['pb_baseline'].append(pb_base)
    results_metrics['pb_peak'].append(pb_pk)
    results_metrics['pb_evoked'].append(pb_evk)
    results_metrics['tr_baseline'].append(tr_base)
    results_metrics['tr_peak'].append(tr_pk)
    results_metrics['tr_evoked'].append(tr_evk)

# 3. Create a DataFrame for easy viewing
df_metrics = pd.DataFrame(results_metrics)
display(df_metrics)

# 4. Plotting the metrics across different subset sizes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Baseline Plot
axes[0].plot(df_metrics['n_events'], df_metrics['pb_baseline'], marker='o', color='black', label='Playback')
axes[0].plot(df_metrics['n_events'], df_metrics['tr_baseline'], marker='o', color='red', label='Tracking')
axes[0].set_title('Baseline (-0.2s to 0s)')
axes[0].set_ylabel('Mean Firing Rate')

# Peak Plot
axes[1].plot(df_metrics['n_events'], df_metrics['pb_peak'], marker='o', color='black', label='Playback')
axes[1].plot(df_metrics['n_events'], df_metrics['tr_peak'], marker='o', color='red', label='Tracking')
axes[1].set_title('Peak Response')

# Total Evoked Response Plot
axes[2].plot(df_metrics['n_events'], df_metrics['pb_evoked'], marker='o', color='black', label='Playback')
axes[2].plot(df_metrics['n_events'], df_metrics['tr_evoked'], marker='o', color='red', label='Tracking')
axes[2].set_title('Total Evoked Response (Peak - Baseline)')

for ax in axes:
    # Set x-axis to logarithmic scale
    ax.set_xscale('log')
    
    # Force the x-ticks to be exactly your n_triggers_list values
    ax.set_xticks(n_triggers_list)
    ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
    
    ax.set_xlabel('Number of Events (n_triggers) [Log Scale]')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import math

# Define the range from 1 to 500
start = 1
stop = 512

# Generate powers of 2: 2^0, 2^1, 2^2, ... up to 500
# We use log2 to determine the maximum exponent
n_triggers_list = [int(round(2**i)) for i in range(int(math.log2(stop)) + 1)]

# Define indices for calculating metrics
# Baseline: 0.2s before sound onset
baseline_bins = int(0.2 / dt)
idx_base_start = n_bins_pre - baseline_bins
idx_base_end = n_bins_pre  # Sound onset (time 0)

# Peak: Max response after sound onset
idx_peak_start = n_bins_pre
idx_peak_end = expected_length

# Define the different "n_triggers" (number of events/trials) you want to evaluate
# n_triggers_list = [1, 2, 3, 5, 7, 10, 20, 30, 50, 70, 100, 200, 300, 500]

# Dictionary to store the results (Means and SEMs)
results_metrics = {
    'n_triggers': [],
    'pb_base_mean': [], 'pb_base_sem': [],
    'pb_peak_mean': [], 'pb_peak_sem': [],
    'pb_evoked_mean': [], 'pb_evoked_sem': [],
    'tr_base_mean': [], 'tr_base_sem': [],
    'tr_peak_mean': [], 'tr_peak_sem': [],
    'tr_evoked_mean': [], 'tr_evoked_sem': []
}

# Helper function to extract and average segments across selected triggers for each channel
def get_subset_channel_means(n_data, event_indices):
    segments = []
    for i in event_indices:
        start, stop = int(i - n_bins_pre), int(i + n_bins_post)
        if 0 <= start and stop <= n_data.shape[-1]:
            seg = n_data[:, start:stop]
            if seg.shape[1] == expected_length:
                segments.append(seg)
    if len(segments) > 0:
        return np.mean(np.stack(segments, axis=0), axis=0) 
    return None

# 2. Main Processing Loop
for n_trig in n_triggers_list:
    means_first_n_pb = []
    means_last_n_tr = []
    
    for n_data, f_data in zip(n_data_s, f_data_s):
        events_pb = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 1))
        events_tr = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 0))
        
        # Subset: First n_trig Playback, Last n_trig Tracking
        events_pb_subset = events_pb[:n_trig] if len(events_pb) >= n_trig else events_pb
        events_tr_subset = events_tr[-n_trig:] if len(events_tr) >= n_trig else events_tr
        
        pb_mean = get_subset_channel_means(n_data, events_pb_subset)
        tr_mean = get_subset_channel_means(n_data, events_tr_subset)
        
        if pb_mean is not None: means_first_n_pb.append(pb_mean)
        if tr_mean is not None: means_last_n_tr.append(tr_mean)

    # Concatenate across all sessions. Shape: (n_total_neurons, expected_length)
    final_pb = np.concatenate(means_first_n_pb, axis=0) if means_first_n_pb else np.array([])
    final_tr = np.concatenate(means_last_n_tr, axis=0) if means_last_n_tr else np.array([])
    
    if final_pb.size == 0 or final_tr.size == 0:
        continue
        
    n_pb_neurons = final_pb.shape[0]
    n_tr_neurons = final_tr.shape[0]
    
    # --- Calculate Metrics for EACH neuron individually for Playback ---
    pb_base_per_neuron = np.mean(final_pb[:, idx_base_start:idx_base_end], axis=1)
    pb_pk_per_neuron = np.max(final_pb[:, idx_peak_start:idx_peak_end], axis=1)
    pb_evk_per_neuron = pb_pk_per_neuron - pb_base_per_neuron
    
    # --- Calculate Metrics for EACH neuron individually for Tracking ---
    tr_base_per_neuron = np.mean(final_tr[:, idx_base_start:idx_base_end], axis=1)
    tr_pk_per_neuron = np.max(final_tr[:, idx_peak_start:idx_peak_end], axis=1)
    tr_evk_per_neuron = tr_pk_per_neuron - tr_base_per_neuron
    
    # Calculate Means and SEMs and store them
    results_metrics['n_triggers'].append(n_trig)
    
    results_metrics['pb_base_mean'].append(np.mean(pb_base_per_neuron))
    results_metrics['pb_base_sem'].append(np.std(pb_base_per_neuron) / np.sqrt(n_pb_neurons))
    results_metrics['pb_peak_mean'].append(np.mean(pb_pk_per_neuron))
    results_metrics['pb_peak_sem'].append(np.std(pb_pk_per_neuron) / np.sqrt(n_pb_neurons))
    results_metrics['pb_evoked_mean'].append(np.mean(pb_evk_per_neuron))
    results_metrics['pb_evoked_sem'].append(np.std(pb_evk_per_neuron) / np.sqrt(n_pb_neurons))
    
    results_metrics['tr_base_mean'].append(np.mean(tr_base_per_neuron))
    results_metrics['tr_base_sem'].append(np.std(tr_base_per_neuron) / np.sqrt(n_tr_neurons))
    results_metrics['tr_peak_mean'].append(np.mean(tr_pk_per_neuron))
    results_metrics['tr_peak_sem'].append(np.std(tr_pk_per_neuron) / np.sqrt(n_tr_neurons))
    results_metrics['tr_evoked_mean'].append(np.mean(tr_evk_per_neuron))
    results_metrics['tr_evoked_sem'].append(np.std(tr_evk_per_neuron) / np.sqrt(n_tr_neurons))




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Assuming 'results_metrics' is defined earlier in your script
df_metrics = pd.DataFrame(results_metrics)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x_vals = df_metrics['n_triggers'].values
n_points = len(x_vals)
center_x = n_points - 0.5  # The exact middle point of the plot

# Create sequential x-coordinates
x_tr = np.arange(n_points)
x_pb = np.arange(n_points) + n_points

# Reverse Tracking data so it plots chronologically toward the center (512 down to 1)
tr_base_mean_rev = df_metrics['tr_base_mean'].values[::-1]
tr_base_sem_rev  = df_metrics['tr_base_sem'].values[::-1]
tr_peak_mean_rev = df_metrics['tr_peak_mean'].values[::-1]
tr_peak_sem_rev  = df_metrics['tr_peak_sem'].values[::-1]
tr_evk_mean_rev  = df_metrics['tr_evoked_mean'].values[::-1]
tr_evk_sem_rev   = df_metrics['tr_evoked_sem'].values[::-1]

# Playback data (normal order)
pb_base_mean = df_metrics['pb_base_mean'].values
pb_base_sem  = df_metrics['pb_base_sem'].values
pb_peak_mean = df_metrics['pb_peak_mean'].values
pb_peak_sem  = df_metrics['pb_peak_sem'].values
pb_evk_mean  = df_metrics['pb_evoked_mean'].values
pb_evk_sem   = df_metrics['pb_evoked_sem'].values

# Create X-tick labels: Reversed for Tracking, Normal for Playback
labels_tr = list(x_vals)[::-1]
labels_pb = list(x_vals)
all_labels = labels_tr + labels_pb

# Formatting helper for the axis
def setup_sequential_axis(ax):
    ax.set_xticks(list(x_tr) + list(x_pb))
    ax.set_xticklabels(all_labels, rotation=45)
    ax.set_xlabel('Number of Triggers')
    # The center dividing line
    ax.axvline(x=center_x, color='gray', linestyle='--', alpha=0.8, zorder=3)
    ax.grid(True, alpha=0.3)
    ax.legend()

# Helper function to plot means, fill SEMs, and draw horizontal dashed lines
def plot_metric_with_fills(ax, tr_m, tr_s, pb_m, pb_s, title, ylabel=None):
    # --- Tracking (Red) ---
    ax.plot(x_tr, tr_m, marker='o', color='red', label='Tracking Mean')
    # ADD LABEL HERE
    ax.fill_between(x_tr, tr_m - tr_s, tr_m + tr_s, color='red', alpha=0.2, label='Tracking SEM')
    ax.hlines(y=tr_m, xmin=x_tr, xmax=center_x, color='red', linestyle='--', alpha=0.4, zorder=1)
    
    # --- Playback (Black) ---
    ax.plot(x_pb, pb_m, marker='o', color='black', label='Playback Mean')
    # ADD LABEL HERE
    ax.fill_between(x_pb, pb_m - pb_s, pb_m + pb_s, color='black', alpha=0.2, label='Playback SEM')
    ax.hlines(y=pb_m, xmin=center_x, xmax=x_pb, color='black', linestyle='--', alpha=0.4, zorder=1)
    
    ax.set_title(title)
    if ylabel:
        ax.set_ylabel(ylabel)
    setup_sequential_axis(ax)

# 3A. Baseline Plot
plot_metric_with_fills(axes[0], tr_base_mean_rev, tr_base_sem_rev, 
                       pb_base_mean, pb_base_sem, 
                       title='Baseline (-0.2s to 0s)', ylabel='Mean Firing Rate')

# 3B. Peak Plot
plot_metric_with_fills(axes[1], tr_peak_mean_rev, tr_peak_sem_rev, 
                       pb_peak_mean, pb_peak_sem, 
                       title='Peak Response')

# 3C. Total Evoked Response Plot
plot_metric_with_fills(axes[2], tr_evk_mean_rev, tr_evk_sem_rev, 
                       pb_evk_mean, pb_evk_sem, 
                       title='Total Evoked Response (Peak - Baseline)')

plt.suptitle(f'Last n Triggers of Tracking vs First n Triggers of Playback', fontsize=16, y=0.99)
plt.tight_layout()
plt.show()

In [ ]:

# Define the range from 1 to 500
start = 1
stop = 512

# Generate powers of 2: 2^0, 2^1, 2^2, ... up to 500
n_triggers_list = [int(round(2**i)) for i in range(int(math.log2(stop)) + 1)]

# Define indices for calculating metrics
baseline_bins = int(0.2 / dt)
idx_base_start = n_bins_pre - baseline_bins
idx_base_end = n_bins_pre  # Sound onset (time 0)

idx_peak_start = n_bins_pre
idx_peak_end = expected_length

# Dictionary to store the results (Means and SEMs)
results_metrics = {
    'n_triggers': [],
    'pb_base_mean': [], 'pb_base_sem': [],
    'pb_peak_mean': [], 'pb_peak_sem': [],
    'pb_evoked_mean': [], 'pb_evoked_sem': [],
    'tr_base_mean': [], 'tr_base_sem': [],
    'tr_peak_mean': [], 'tr_peak_sem': [],
    'tr_evoked_mean': [], 'tr_evoked_sem': []
}

# Helper function
def get_subset_channel_means(n_data, event_indices):
    segments = []
    for i in event_indices:
        start, stop = int(i - n_bins_pre), int(i + n_bins_post)
        if 0 <= start and stop <= n_data.shape[-1]:
            seg = n_data[:, start:stop]
            if seg.shape[1] == expected_length:
                segments.append(seg)
    if len(segments) > 0:
        return np.mean(np.stack(segments, axis=0), axis=0) 
    return None

# 2. Main Processing Loop
for n_trig in n_triggers_list:
    means_first_n_pb = []
    means_last_n_tr = []
    
    for n_data, f_data in zip(n_data_s, f_data_s):
        events_pb = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 1))
        events_tr = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 0))
        
        events_pb_subset = events_pb[:n_trig] if len(events_pb) >= n_trig else events_pb
        events_tr_subset = events_tr[-n_trig:] if len(events_tr) >= n_trig else events_tr
        
        pb_mean = get_subset_channel_means(n_data, events_pb_subset)
        tr_mean = get_subset_channel_means(n_data, events_tr_subset)
        
        if pb_mean is not None: means_first_n_pb.append(pb_mean)
        if tr_mean is not None: means_last_n_tr.append(tr_mean)

    final_pb = np.concatenate(means_first_n_pb, axis=0) if means_first_n_pb else np.array([])
    final_tr = np.concatenate(means_last_n_tr, axis=0) if means_last_n_tr else np.array([])
    
    if final_pb.size == 0 or final_tr.size == 0:
        continue
        
    n_pb_neurons = final_pb.shape[0]
    n_tr_neurons = final_tr.shape[0]
    
    # =========================================================
    # APPLY Z-SCORE (按每个神经元独立计算 Z-score)
    # =========================================================
    # 为了避免除以 0，给标准差加上一个极小值 (1e-8)
    
    # Playback Z-score (基于整个时段计算均值和标准差)
    pb_mean_time = np.mean(final_pb, axis=1, keepdims=True)
    pb_std_time = np.std(final_pb, axis=1, keepdims=True) + 1e-8
    final_pb_z = (final_pb - pb_mean_time) / pb_std_time
    
    # Tracking Z-score
    tr_mean_time = np.mean(final_tr, axis=1, keepdims=True)
    tr_std_time = np.std(final_tr, axis=1, keepdims=True) + 1e-8
    final_tr_z = (final_tr - tr_mean_time) / tr_std_time

    # --- 基于 Z-scored 数据计算 Playback 的 Metrics ---
    pb_base_per_neuron = np.mean(final_pb_z[:, idx_base_start:idx_base_end], axis=1)
    pb_pk_per_neuron = np.max(final_pb_z[:, idx_peak_start:idx_peak_end], axis=1)
    pb_evk_per_neuron = pb_pk_per_neuron - pb_base_per_neuron
    
    # --- 基于 Z-scored 数据计算 Tracking 的 Metrics ---
    tr_base_per_neuron = np.mean(final_tr_z[:, idx_base_start:idx_base_end], axis=1)
    tr_pk_per_neuron = np.max(final_tr_z[:, idx_peak_start:idx_peak_end], axis=1)
    tr_evk_per_neuron = tr_pk_per_neuron - tr_base_per_neuron
    
    # Calculate Means and SEMs and store them
    results_metrics['n_triggers'].append(n_trig)
    
    results_metrics['pb_base_mean'].append(np.mean(pb_base_per_neuron))
    results_metrics['pb_base_sem'].append(np.std(pb_base_per_neuron) / np.sqrt(n_pb_neurons))
    results_metrics['pb_peak_mean'].append(np.mean(pb_pk_per_neuron))
    results_metrics['pb_peak_sem'].append(np.std(pb_pk_per_neuron) / np.sqrt(n_pb_neurons))
    results_metrics['pb_evoked_mean'].append(np.mean(pb_evk_per_neuron))
    results_metrics['pb_evoked_sem'].append(np.std(pb_evk_per_neuron) / np.sqrt(n_pb_neurons))
    
    results_metrics['tr_base_mean'].append(np.mean(tr_base_per_neuron))
    results_metrics['tr_base_sem'].append(np.std(tr_base_per_neuron) / np.sqrt(n_tr_neurons))
    results_metrics['tr_peak_mean'].append(np.mean(tr_pk_per_neuron))
    results_metrics['tr_peak_sem'].append(np.std(tr_pk_per_neuron) / np.sqrt(n_tr_neurons))
    results_metrics['tr_evoked_mean'].append(np.mean(tr_evk_per_neuron))
    results_metrics['tr_evoked_sem'].append(np.std(tr_evk_per_neuron) / np.sqrt(n_tr_neurons))


In [ ]:

# =========================================================
# PLOTTING
# =========================================================
df_metrics = pd.DataFrame(results_metrics)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x_vals = df_metrics['n_triggers'].values
n_points = len(x_vals)
center_x = n_points - 0.5

x_tr = np.arange(n_points)
x_pb = np.arange(n_points) + n_points

# Reverse Tracking data
tr_base_mean_rev = df_metrics['tr_base_mean'].values[::-1]
tr_base_sem_rev  = df_metrics['tr_base_sem'].values[::-1]
tr_peak_mean_rev = df_metrics['tr_peak_mean'].values[::-1]
tr_peak_sem_rev  = df_metrics['tr_peak_sem'].values[::-1]
tr_evk_mean_rev  = df_metrics['tr_evoked_mean'].values[::-1]
tr_evk_sem_rev   = df_metrics['tr_evoked_sem'].values[::-1]

# Playback data
pb_base_mean = df_metrics['pb_base_mean'].values
pb_base_sem  = df_metrics['pb_base_sem'].values
pb_peak_mean = df_metrics['pb_peak_mean'].values
pb_peak_sem  = df_metrics['pb_peak_sem'].values
pb_evk_mean  = df_metrics['pb_evoked_mean'].values
pb_evk_sem   = df_metrics['pb_evoked_sem'].values

labels_tr = list(x_vals)[::-1]
labels_pb = list(x_vals)
all_labels = labels_tr + labels_pb

def setup_sequential_axis(ax):
    ax.set_xticks(list(x_tr) + list(x_pb))
    ax.set_xticklabels(all_labels, rotation=45)
    ax.set_xlabel('Number of Triggers')
    ax.axvline(x=center_x, color='gray', linestyle='--', alpha=0.8, zorder=3)
    ax.grid(True, alpha=0.3)
    ax.legend()

def plot_metric_with_fills(ax, tr_m, tr_s, pb_m, pb_s, title, ylabel=None):
    # Tracking (Red)
    ax.plot(x_tr, tr_m, marker='o', color='red', label='Tracking Mean')
    ax.fill_between(x_tr, tr_m - tr_s, tr_m + tr_s, color='red', alpha=0.2, label='Tracking SEM')
    ax.hlines(y=tr_m, xmin=x_tr, xmax=center_x, color='red', linestyle='--', alpha=0.4, zorder=1)
    
    # Playback (Black)
    ax.plot(x_pb, pb_m, marker='o', color='black', label='Playback Mean')
    ax.fill_between(x_pb, pb_m - pb_s, pb_m + pb_s, color='black', alpha=0.2, label='Playback SEM')
    ax.hlines(y=pb_m, xmin=center_x, xmax=x_pb, color='black', linestyle='--', alpha=0.4, zorder=1)
    
    ax.set_title(title)
    if ylabel:
        ax.set_ylabel(ylabel)
    setup_sequential_axis(ax)

# 3A. Baseline Plot
plot_metric_with_fills(axes[0], tr_base_mean_rev, tr_base_sem_rev, 
                       pb_base_mean, pb_base_sem, 
                       title='Baseline Z-Score (-0.2s to 0s)', ylabel='Z-Score')

# 3B. Peak Plot
plot_metric_with_fills(axes[1], tr_peak_mean_rev, tr_peak_sem_rev, 
                       pb_peak_mean, pb_peak_sem, 
                       title='Peak Response Z-Score', ylabel='Z-Score')

# 3C. Total Evoked Response Plot
plot_metric_with_fills(axes[2], tr_evk_mean_rev, tr_evk_sem_rev, 
                       pb_evk_mean, pb_evk_sem, 
                       title='Total Evoked Response (Peak - Baseline Z-Score)', ylabel='Z-Score')

plt.suptitle(f'Last n Triggers of Tracking vs First n Triggers of Playback (Z-Scored)', fontsize=16, y=0.99)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# --- CUSTOMIZABLE VARIABLES ---
n_per_group = 1    # Number of triggers per group
n_groups = 3        # Number of chronological groups per condition
# ==========================================

# Generate keys dynamically based on variables
tr_keys = []
for i in range(n_groups, 0, -1):
    start_idx = i * n_per_group
    end_idx = (i - 1) * n_per_group + 1
    tr_keys.append(f'Tracking: Last {start_idx}-{end_idx}')

pb_keys = []
for i in range(n_groups):
    start_idx = i * n_per_group + 1
    end_idx = (i + 1) * n_per_group
    pb_keys.append(f'Playback: First {start_idx}-{end_idx}')

all_keys = tr_keys + pb_keys
all_data = {key: [] for key in all_keys}

# Helper function to extract and average segments
def get_subset_channel_means(n_data, event_indices):
    segments = []
    for i in event_indices:
        start, stop = int(i - n_bins_pre), int(i + n_bins_post)
        if 0 <= start and stop <= n_data.shape[-1]:
            seg = n_data[:, start:stop]
            if seg.shape[1] == expected_length:
                segments.append(seg)
    if len(segments) > 0:
        return np.mean(np.stack(segments, axis=0), axis=0)
    return None

# 2. Main Processing Loop
for n_data, f_data in zip(n_data_s, f_data_s):
    events_pb = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 1))
    events_tr = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 0))
    
    # --- Tracking: Slice chronological chunks from the end ---
    total_tr_needed = n_groups * n_per_group
    tr_chunks = []
    
    # Only process if the session has enough tracking events
    if len(events_tr) >= total_tr_needed:
        for i in range(n_groups, 0, -1):
            start = -i * n_per_group
            end = -(i - 1) * n_per_group if i > 1 else None
            tr_chunks.append(events_tr[start:end])
    else:
        tr_chunks = [[] for _ in range(n_groups)] 
        
    for key, evs in zip(tr_keys, tr_chunks):
        if len(evs) > 0:
            mean_trace = get_subset_channel_means(n_data, evs)
            if mean_trace is not None:
                all_data[key].append(mean_trace)

    # --- Playback: Slice chronological chunks from the beginning ---
    total_pb_needed = n_groups * n_per_group
    pb_chunks = []
    
    # Only process if the session has enough playback events
    if len(events_pb) >= total_pb_needed:
        for i in range(n_groups):
            start = i * n_per_group
            end = (i + 1) * n_per_group
            pb_chunks.append(events_pb[start:end])
    else:
        pb_chunks = [[] for _ in range(n_groups)]
        
    for key, evs in zip(pb_keys, pb_chunks):
        if len(evs) > 0:
            mean_trace = get_subset_channel_means(n_data, evs)
            if mean_trace is not None:
                all_data[key].append(mean_trace)

# 3. Plotting in a Dynamic Grid
# Create a 2 x n_groups grid (Top row: Tracking, Bottom row: Playback)
fig, axes = plt.subplots(1, n_groups * 2 , figsize=(20 * n_groups, 8), sharey=True, sharex=True)

# Ensure axes is a 2D array even if n_groups=1
axes = np.array(axes).reshape(2, n_groups)

# Generate dynamic colors (from light to dark)
tr_colors = [plt.cm.Reds(x) for x in np.linspace(0.4, 0.9, n_groups)]
pb_colors = [plt.cm.Greys(x) for x in np.linspace(0.4, 0.9, n_groups)]

# --- Plot Tracking (Top Row) ---
for i, (key, color) in enumerate(zip(tr_keys, tr_colors)):
    ax = axes[0, i]
    if len(all_data[key]) > 0:
        final_dat = np.concatenate(all_data[key], axis=0)
        mean_dat = np.mean(final_dat, axis=0)
        sem_dat = np.std(final_dat, axis=0) / np.sqrt(final_dat.shape[0])
        
        ax.plot(time, mean_dat, color=color, linewidth=2.5, label=f'Neurons: n={final_dat.shape[0]}')
        ax.fill_between(time, mean_dat - sem_dat, mean_dat + sem_dat, color=color, alpha=0.15)
        ax.legend(loc='upper right')
        
    ax.axvline(0, color='blue', linestyle='--', alpha=0.5, label='Sound Onset' if i==0 else "")
    ax.set_title(key, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    if i == 0: ax.set_ylabel('Mean Firing Rate\n[Tracking]', fontsize=12)

# --- Plot Playback (Bottom Row) ---
for i, (key, color) in enumerate(zip(pb_keys, pb_colors)):
    ax = axes[1, i]
    if len(all_data[key]) > 0:
        final_dat = np.concatenate(all_data[key], axis=0)
        mean_dat = np.mean(final_dat, axis=0)
        sem_dat = np.std(final_dat, axis=0) / np.sqrt(final_dat.shape[0])
        
        ax.plot(time, mean_dat, color=color, linewidth=2.5, label=f'Neurons: n={final_dat.shape[0]}')
        ax.fill_between(time, mean_dat - sem_dat, mean_dat + sem_dat, color=color, alpha=0.15)
        ax.legend(loc='upper right')
        
    ax.axvline(0, color='blue', linestyle='--', alpha=0.5)
    ax.set_title(key, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('Time relative to event (s)', fontsize=12)
    if i == 0: ax.set_ylabel('Mean Firing Rate\n[Playback]', fontsize=12)

plt.suptitle(f'Chronological PSTH sEvolution ({n_per_group} triggers per group)', fontsize=16, y=1.02)
plt.tight_layout()  
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# --- CUSTOMIZABLE VARIABLES ---
n_per_group = 5    # Number of triggers per group
n_groups = 3        # Number of chronological groups per condition
# ==========================================

# Generate keys dynamically based on variables
tr_keys = []
for i in range(n_groups, 0, -1):
    start_idx = i * n_per_group
    end_idx = (i - 1) * n_per_group + 1
    tr_keys.append(f'Tracking: Last {start_idx}-{end_idx}')

pb_keys = []
for i in range(n_groups):
    start_idx = i * n_per_group + 1
    end_idx = (i + 1) * n_per_group
    pb_keys.append(f'Playback: First {start_idx}-{end_idx}')

all_keys = tr_keys + pb_keys
all_data = {key: [] for key in all_keys}

# Helper function to extract and average segments
def get_subset_channel_means(n_data, event_indices):
    segments = []
    for i in event_indices:
        start, stop = int(i - n_bins_pre), int(i + n_bins_post)
        if 0 <= start and stop <= n_data.shape[-1]:
            seg = n_data[:, start:stop]
            if seg.shape[1] == expected_length:
                segments.append(seg)
    if len(segments) > 0:
        return np.mean(np.stack(segments, axis=0), axis=0)
    return None

# 2. Main Processing Loop
for n_data, f_data in zip(n_data_s, f_data_s):
    events_pb = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 1))
    events_tr = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 0))
    
    # --- Tracking: Slice chronological chunks from the end ---
    total_tr_needed = n_groups * n_per_group
    tr_chunks = []
    
    # Only process if the session has enough tracking events
    if len(events_tr) >= total_tr_needed:
        for i in range(n_groups, 0, -1):
            start = -i * n_per_group
            end = -(i - 1) * n_per_group if i > 1 else None
            tr_chunks.append(events_tr[start:end])
    else:
        tr_chunks = [[] for _ in range(n_groups)] 
        
    for key, evs in zip(tr_keys, tr_chunks):
        if len(evs) > 0:
            mean_trace = get_subset_channel_means(n_data, evs)
            if mean_trace is not None:
                all_data[key].append(mean_trace)

    # --- Playback: Slice chronological chunks from the beginning ---
    total_pb_needed = n_groups * n_per_group
    pb_chunks = []
    
    # Only process if the session has enough playback events
    if len(events_pb) >= total_pb_needed:
        for i in range(n_groups):
            start = i * n_per_group
            end = (i + 1) * n_per_group
            pb_chunks.append(events_pb[start:end])
    else:
        pb_chunks = [[] for _ in range(n_groups)]
        
    for key, evs in zip(pb_keys, pb_chunks):
        if len(evs) > 0:
            mean_trace = get_subset_channel_means(n_data, evs)
            if mean_trace is not None:
                all_data[key].append(mean_trace)

# 3. Plotting in a Dynamic Grid
# Create a 2 x n_groups grid (Top row: Tracking, Bottom row: Playback)
fig, axes = plt.subplots(1, n_groups * 2 , figsize=(20 * n_groups, 8), sharey=True, sharex=True)

# Ensure axes is a 2D array even if n_groups=1
axes = np.array(axes).reshape(2, n_groups)

# Generate dynamic colors (from light to dark)
tr_colors = [plt.cm.Reds(x) for x in np.linspace(0.4, 0.9, n_groups)]
pb_colors = [plt.cm.Greys(x) for x in np.linspace(0.4, 0.9, n_groups)]

# --- Plot Tracking (Top Row) ---
for i, (key, color) in enumerate(zip(tr_keys, tr_colors)):
    ax = axes[0, i]
    n_neurons = 0 # Default to 0 if no data
    if len(all_data[key]) > 0:
        final_dat = np.concatenate(all_data[key], axis=0)
        mean_dat = np.mean(final_dat, axis=0)
        sem_dat = np.std(final_dat, axis=0) / np.sqrt(final_dat.shape[0])
        n_neurons = final_dat.shape[0] # <--- Extract number of neurons here
        
        ax.plot(time, mean_dat, color=color, linewidth=2.5, label='Mean FR')
        ax.fill_between(time, mean_dat - sem_dat, mean_dat + sem_dat, color=color, alpha=0.15)
        
        # Add a text box directly inside the plot
        ax.text(0.05, 0.95, f'n = {n_neurons} neurons', transform=ax.transAxes, 
                fontsize=12, verticalalignment='top', 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor=color))
        
    ax.axvline(0, color='blue', linestyle='--', alpha=0.5, label='Sound Onset' if i==0 else "")
    
    # Update the title to include the neuron count
    ax.set_title(f"{key}\n(n={n_neurons})", fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    if i == 0: ax.set_ylabel('Mean Firing Rate\n[Tracking]', fontsize=12)
    ax.legend(loc='lower right') # Moved legend so it doesn't overlap the text box

# --- Plot Playback (Bottom Row) ---
for i, (key, color) in enumerate(zip(pb_keys, pb_colors)):
    ax = axes[1, i]
    n_neurons = 0
    if len(all_data[key]) > 0:
        final_dat = np.concatenate(all_data[key], axis=0)
        mean_dat = np.mean(final_dat, axis=0)
        sem_dat = np.std(final_dat, axis=0) / np.sqrt(final_dat.shape[0])
        n_neurons = final_dat.shape[0] # <--- Extract number of neurons here
        
        ax.plot(time, mean_dat, color=color, linewidth=2.5, label='Mean FR')
        ax.fill_between(time, mean_dat - sem_dat, mean_dat + sem_dat, color=color, alpha=0.15)
        
        # Add a text box directly inside the plot
        ax.text(0.05, 0.95, f'n = {n_neurons} neurons', transform=ax.transAxes, 
                fontsize=12, verticalalignment='top', 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor=color))
        
    ax.axvline(0, color='blue', linestyle='--', alpha=0.5)
    
    # Update the title to include the neuron count
    ax.set_title(f"{key}\n(n={n_neurons})", fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('Time relative to event (s)', fontsize=12)
    if i == 0: ax.set_ylabel('Mean Firing Rate\n[Playback]', fontsize=12)
    ax.legend(loc='lower right')

plt.suptitle(f'Chronological PSTH Evolution ({n_per_group} triggers per group)', fontsize=18, y=1.05)
plt.tight_layout()  
plt.show()

In [ ]:
# Set your thresholds (matching your last cell)
n_per_group = 5    
n_groups = 3        
total_needed = n_groups * n_per_group  # 15 events needed

print(f"--- Diagnostic: Checking for Session Mismatches (Threshold: {total_needed} events) ---\n")

# Keep track of totals
total_pb_neurons = 0
total_tr_neurons = 0
mismatched_sessions = []

for i, (n_data, f_data) in enumerate(zip(n_data_s, f_data_s)):
    # 1. Count the events
    events_pb = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 1))
    events_tr = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 0))
    
    # 2. Check against your threshold
    has_enough_pb = len(events_pb) >= total_needed
    has_enough_tr = len(events_tr) >= total_needed
    
    # Get session name safely (using your all_sessions_path variable from Cell 3)
    try:
        session_name = os.path.basename(all_sessions_path[i])
    except NameError:
        session_name = f"Session Index {i}"
        
    num_neurons = n_data.shape[0]
    
    # 3. Flag the mismatches
    if has_enough_pb and not has_enough_tr:
        print(f"⚠️ MISMATCH | {session_name}:")
        print(f"   -> Kept for Playback ({len(events_pb)} events)")
        print(f"   -> EXCLUDED for Tracking ({len(events_tr)} events)")
        print(f"   -> Result: +{num_neurons} neurons added to Playback ONLY.\n")
        mismatched_sessions.append(session_name)
        total_pb_neurons += num_neurons
        
    elif has_enough_tr and not has_enough_pb:
        print(f"⚠️ MISMATCH | {session_name}:")
        print(f"   -> Kept for Tracking ({len(events_tr)} events)")
        print(f"   -> EXCLUDED for Playback ({len(events_pb)} events)")
        print(f"   -> Result: +{num_neurons} neurons added to Tracking ONLY.\n")
        mismatched_sessions.append(session_name)
        total_tr_neurons += num_neurons
        
    # 4. Tally matches (for your own sanity check)
    elif has_enough_pb and has_enough_tr:
        total_pb_neurons += num_neurons
        total_tr_neurons += num_neurons

print("--- Summary ---")
print(f"Total mismatched sessions: {len(mismatched_sessions)}")
print(f"Final Expected Playback Neurons: {total_pb_neurons}")
print(f"Final Expected Tracking Neurons: {total_tr_neurons}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# --- CUSTOMIZABLE VARIABLES ---
n_per_group = 100    # Number of triggers per group
n_groups = 3        # Number of chronological groups per condition
# ==========================================

total_needed = n_groups * n_per_group

# Define indices for calculating metrics
baseline_bins = int(0.2 / dt)
idx_base_start = n_bins_pre - baseline_bins
idx_base_end = n_bins_pre  # Sound onset (time 0)
idx_peak_start = n_bins_pre
idx_peak_end = expected_length

# Generate x-axis labels for the chunks
# Tracking: Last chunks (15-11, 10-6, 5-1) chronological
# Playback: First chunks (1-5, 6-10, 11-15) chronological
x_labels_tr = [f"TR: Last {i*n_per_group}-{(i-1)*n_per_group+1}" for i in range(n_groups, 0, -1)]
x_labels_pb = [f"PB: First {i*n_per_group+1}-{(i+1)*n_per_group}" for i in range(n_groups)]
all_x_labels = x_labels_tr + x_labels_pb

# Containers for the chunks
tr_data_chunks = [[] for _ in range(n_groups)]
pb_data_chunks = [[] for _ in range(n_groups)]

# Helper function
def get_subset_channel_means(n_data, event_indices):
    segments = []
    for i in event_indices:
        start, stop = int(i - n_bins_pre), int(i + n_bins_post)
        if 0 <= start and stop <= n_data.shape[-1]:
            seg = n_data[:, start:stop]
            if seg.shape[1] == expected_length:
                segments.append(seg)
    if len(segments) > 0:
        return np.mean(np.stack(segments, axis=0), axis=0)
    return None

# 1. Main Processing Loop
for n_data, f_data in zip(n_data_s, f_data_s):
    events_pb = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 1))
    events_tr = np.flatnonzero((f_data['Frequency_changes'] == 1) & (f_data['Condition'] == 0))
    
    # STRICT INCLUSION: Only process if the session has enough events for BOTH conditions
    if len(events_tr) >= total_needed and len(events_pb) >= total_needed:
        
        # --- Tracking Chunks (Chronological from the end) ---
        for chunk_idx, i in enumerate(range(n_groups, 0, -1)):
            start = -i * n_per_group
            end = -(i - 1) * n_per_group if i > 1 else None
            evs = events_tr[start:end]
            mean_trace = get_subset_channel_means(n_data, evs)
            if mean_trace is not None:
                tr_data_chunks[chunk_idx].append(mean_trace)

        # --- Playback Chunks (Chronological from the start) ---
        for chunk_idx, i in enumerate(range(n_groups)):
            start = i * n_per_group
            end = (i + 1) * n_per_group
            evs = events_pb[start:end]
            mean_trace = get_subset_channel_means(n_data, evs)
            if mean_trace is not None:
                pb_data_chunks[chunk_idx].append(mean_trace)

# 2. Compute Z-Scores and Metrics
def process_metrics(data_chunks):
    means = {'base': [], 'peak': [], 'evoked': []}
    sems = {'base': [], 'peak': [], 'evoked': []}
    n_neurons = 0
    
    for chunk in data_chunks:
        if len(chunk) > 0:
            final_dat = np.concatenate(chunk, axis=0)
            n_neurons = final_dat.shape[0]
            
            # Z-Score per neuron
            mean_time = np.mean(final_dat, axis=1, keepdims=True)
            std_time = np.std(final_dat, axis=1, keepdims=True) + 1e-8
            z_dat = (final_dat - mean_time) / std_time
            
            # Extract Metrics
            base = np.mean(z_dat[:, idx_base_start:idx_base_end], axis=1)
            peak = np.max(z_dat[:, idx_peak_start:idx_peak_end], axis=1)
            evoked = peak - base
            
            # Store Means and SEMs
            means['base'].append(np.mean(base))
            sems['base'].append(np.std(base) / np.sqrt(n_neurons))
            
            means['peak'].append(np.mean(peak))
            sems['peak'].append(np.std(peak) / np.sqrt(n_neurons))
            
            means['evoked'].append(np.mean(evoked))
            sems['evoked'].append(np.std(evoked) / np.sqrt(n_neurons))
        else:
            for k in means:
                means[k].append(np.nan)
                sems[k].append(np.nan)
                
    return means, sems, n_neurons

tr_means, tr_sems, n_tr_neurons = process_metrics(tr_data_chunks)
pb_means, pb_sems, n_pb_neurons = process_metrics(pb_data_chunks)

# 3. Plotting
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Define sequential x-coordinates for Tracking (left) and Playback (right)
x_tr = np.arange(n_groups)
x_pb = np.arange(n_groups) + n_groups
all_x_vals = list(x_tr) + list(x_pb)
divider_x = n_groups - 0.5  # The midpoint between Tracking and Playback

metrics = ['base', 'peak', 'evoked']
titles = ['Baseline Z-Score (-0.2s to 0s)', 'Peak Response Z-Score', 'Total Evoked Response (Peak - Baseline)']

for i, metric in enumerate(metrics):
    ax = axes[i]
    
    # Plot Tracking (Red, Left side)
    tr_m = np.array(tr_means[metric])
    tr_s = np.array(tr_sems[metric])
    ax.errorbar(x_tr, tr_m, yerr=tr_s, color='red', marker='o', markersize=8, 
                linewidth=2.5, label=f'Tracking (n={n_tr_neurons})', capsize=5)
    
    # Plot Playback (Black, Right side)
    pb_m = np.array(pb_means[metric])
    pb_s = np.array(pb_sems[metric])
    ax.errorbar(x_pb, pb_m, yerr=pb_s, color='black', marker='o', markersize=8, 
                linewidth=2.5, label=f'Playback (n={n_pb_neurons})', capsize=5)
    
    # Formatting
    ax.axvline(x=divider_x, color='gray', linestyle='--', alpha=0.8, zorder=1) # Visual separator
    ax.set_title(titles[i], fontsize=14, fontweight='bold')
    ax.set_xticks(all_x_vals)
    ax.set_xticklabels(all_x_labels, rotation=45, ha='right') # Rotated for readability
    ax.grid(True, alpha=0.3)
    
    if i == 0:
        ax.set_ylabel('Mean Z-Score', fontsize=12)
    ax.legend(loc='best')

plt.suptitle(f'Chronological Z-Score Metrics ({n_per_group} triggers/chunk)', fontsize=18, y=1.05)
plt.tight_layout()
plt.show()